# Audiencias Agencia Digital — Laboratorio Económico
Notebook hermano de `12_marketing_audiences_economics_lab.ipynb`, limitado a los medios de la agencia tercerizada. El objetivo es leer **escala, calidad, concentración, cohortes, hipótesis y RMKT** como economista, sin confundir asociación con causalidad.

> El análisis central no extrae PII. Las bases del Excel se consultan solo al final y únicamente con `apto_entrega_externa = TRUE`.


## 1. Universo y configuración
Los medios viven en `config/marketing_agency_channels.yml`; inicialmente: Facebook, Tik Tok, Instagram, Meta y Google. Para agregar alias no se modifica SQL ni gráficos.


In [ ]:
from pathlib import Path
from datetime import datetime
import re, sys, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

warnings.filterwarnings("ignore", category=FutureWarning)

def root(start):
    for x in (start, *start.parents):
        if (x / "pyproject.toml").exists() and (x / "src").exists(): return x
    raise RuntimeError("Abra el notebook dentro de bd_replica_crm")

ROOT=root(Path.cwd().resolve()); sys.path.insert(0, str(ROOT / "src"))
from replica_cygnus.connections import connect_postgres
from replica_cygnus.settings import load_settings

cfg=yaml.safe_load((ROOT/"config"/"marketing_agency_channels.yml").read_text(encoding="utf-8"))
norm=lambda x: re.sub(r"\s+"," ",str(x or "").strip().lower())
alias={norm(v):c["label"] for c in cfg["channels"] for v in c["match_values"]}
MEDIA=sorted(alias); LABELS=[c["label"] for c in cfg["channels"]]

def q(sql, params=None):
    with connect_postgres(load_settings(require_source=False)) as cn:
        with cn.transaction(), cn.cursor() as cur:
            cur.execute("SET TRANSACTION READ ONLY"); cur.execute(sql, params)
            return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

SQL="""
SELECT COALESCE(NULLIF(proyecto,''),'Sin proyecto') proyecto,
 regexp_replace(lower(trim(COALESCE(medio_captacion,''))), '\\s+', ' ', 'g') medio,
 fecha_cliente, etapa_maxima, positiva::int positiva,
 negativa_sin_avance::int negativa_sin_avance, contacto_valido::int contacto_valido,
 apto_entrega_externa::int apto_entrega_externa
FROM analytics.marketing_audiencias
WHERE regexp_replace(lower(trim(COALESCE(medio_captacion,''))), '\\s+', ' ', 'g') = ANY(%s)
"""
aud=q(SQL,(MEDIA,)); aud["canal"]=aud["medio"].map(alias)
aud["fecha_cliente"]=pd.to_datetime(aud["fecha_cliente"],errors="coerce")
aud["negativa_activable"]=((aud.negativa_sin_avance==1)&(aud.apto_entrega_externa==1)).astype(int)
assert not aud.empty and aud.canal.notna().all()
print(f"Personas agencia: {len(aud):,} | PII en análisis: NO")


## 2. Estadística, concentración e incertidumbre
Separa **volumen** de **tasa positiva**. Una tasa alta en una base pequeña no implica mayor impacto económico. El HHI resume dependencia de pocos medios y Wilson 95% evita leer rankings puntuales como certezas.


In [ ]:
ch=(aud.groupby("canal").agg(personas=("positiva","size"),positivas=("positiva","sum"),
 tasa_positiva=("positiva","mean"),negativas_rmkt=("negativa_sin_avance","sum"),
 negativas_activables=("negativa_activable","sum"),contactabilidad=("contacto_valido","mean"),
 aptitud_externa=("apto_entrega_externa","mean")).sort_values("personas",ascending=False))
ch["participacion"]=ch.personas/ch.personas.sum(); hhi=float((ch.participacion**2).sum())
ci=ch.apply(lambda r: proportion_confint(int(r.positivas),int(r.personas),method="wilson"),axis=1)
ch["ci95_low"]=[x[0] for x in ci]; ch["ci95_high"]=[x[1] for x in ci]
portfolio=float(aud.positiva.mean()); ch["brecha_pp"]=(ch.tasa_positiva-portfolio)*100

display(ch.reset_index()); print(f"Tasa agencia={portfolio:.2%} | HHI={hhi:.3f}")
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(ch.index,ch.personas); ax.set(title="Escala por medio",ylabel="Personas"); plt.xticks(rotation=20); plt.show()
fig,ax=plt.subplots(figsize=(10,4)); x=np.arange(len(ch)); y=ch.tasa_positiva.to_numpy();
ax.errorbar(x,y,yerr=np.vstack([y-ch.ci95_low.to_numpy(),ch.ci95_high.to_numpy()-y]),fmt="o",capsize=5)
ax.axhline(portfolio,ls="--",label="Promedio agencia"); ax.set_xticks(x,ch.index,rotation=20); ax.set(title="Tasa positiva + Wilson 95%",ylabel="Tasa"); ax.legend(); plt.show()


## 3. Hipótesis y matriz proyecto × medio
**H0:** la tasa positiva es independiente del medio. Se usa χ² global y comparaciones de proporciones contra el canal de mayor volumen con ajuste de Holm. Esto detecta asociación, no efecto causal.


In [ ]:
ct=pd.crosstab(aud.canal,aud.positiva).reindex(columns=[0,1],fill_value=0)
chi2,p_global,dof,_=chi2_contingency(ct)
benchmark=ch.index[0]; tests=[]
for c in ch.index[1:]:
    z,p=proportions_ztest([int(ch.loc[c,"positivas"]),int(ch.loc[benchmark,"positivas"])],
                          [int(ch.loc[c,"personas"]),int(ch.loc[benchmark,"personas"])])
    tests.append([benchmark,c,(ch.loc[c,"tasa_positiva"]-ch.loc[benchmark,"tasa_positiva"])*100,z,p])
pair=pd.DataFrame(tests,columns=["benchmark","challenger","diferencia_pp","z","p_value"])
if len(pair): pair["p_holm"]=multipletests(pair.p_value,method="holm")[1]
print(f"Chi²={chi2:.2f} | p={p_global:.4g} | gl={dof}"); display(pair)

pc=(aud.groupby(["proyecto","canal"]).agg(personas=("positiva","size"),tasa=("positiva","mean"),
 negativas_activables=("negativa_activable","sum")).reset_index())
top=pc.groupby("proyecto").personas.sum().nlargest(15).index
m=pc[pc.proyecto.isin(top)].pivot(index="proyecto",columns="canal",values="tasa").reindex(columns=LABELS)
display(m)
fig,ax=plt.subplots(figsize=(10,7)); im=ax.imshow(m.to_numpy(float),aspect="auto",vmin=0,vmax=max(.05,np.nanmax(m.to_numpy(float))))
ax.set_xticks(range(len(m.columns)),m.columns,rotation=25,ha="right"); ax.set_yticks(range(len(m.index)),m.index); ax.set_title("Tasa positiva: proyecto × medio"); fig.colorbar(im,ax=ax); plt.show()


## 4. Cohortes y RMKT activable
Las cohortes recientes tienen menos tiempo de maduración. El stock útil para una agencia no es toda `negativa_sin_avance`, sino solo la intersección con `apto_entrega_externa = TRUE`.


In [ ]:
co=aud.dropna(subset=["fecha_cliente"]).copy(); co["cohorte"]=co.fecha_cliente.dt.to_period("M").dt.to_timestamp()
coh=(co.groupby(["cohorte","canal"]).agg(personas=("positiva","size"),tasa_positiva=("positiva","mean"),
 negativas_activables=("negativa_activable","sum")).reset_index())
recent=coh[coh.cohorte>=coh.cohorte.max()-pd.DateOffset(months=23)].copy(); recent.loc[recent.personas<40,"tasa_positiva"]=np.nan
fig,ax=plt.subplots(figsize=(11,5))
for c,d in recent.groupby("canal"): ax.plot(d.cohorte,d.tasa_positiva,marker="o",label=c)
ax.set(title="Cohortes: tasa positiva",ylabel="Tasa"); ax.legend(ncol=3); plt.show()
rmkt=pc.sort_values("negativas_activables",ascending=False); display(rmkt.head(30))


## 5. Excel ejecutivo aesthetic + bases positivas y negativas
Genera `outputs/marketing_agency_audiences/audiencias_agencia_ejecutivo_*.xlsx`. Las hojas operativas con nombre/celular/email se consultan recién aquí, desde las vistas existentes, y **solo** cuando `apto_entrega_externa = TRUE`.


In [ ]:
POS="""SELECT "Nombres" nombres,"Apellidos" apellidos,"Telefono_Celular" telefono_celular,
 "Correo_electronico" correo_electronico,medio_captacion,proyecto,etapa_maxima,fecha_ultimo_avance
 FROM analytics.v_marketing_positiva WHERE apto_entrega_externa=TRUE
 AND regexp_replace(lower(trim(COALESCE(medio_captacion,''))), '\\s+', ' ', 'g')=ANY(%s)"""
NEG="""SELECT "Nombres" nombres,"Apellidos" apellidos,"Telefono_Celular" telefono_celular,
 "Correo_electronico" correo_electronico,medio_captacion,proyecto,etapa_maxima,fecha_cliente
 FROM analytics.v_marketing_negativa_sin_avance WHERE apto_entrega_externa=TRUE
 AND regexp_replace(lower(trim(COALESCE(medio_captacion,''))), '\\s+', ' ', 'g')=ANY(%s)"""
base_pos=q(POS,(MEDIA,)); base_neg=q(NEG,(MEDIA,))
out=ROOT/"outputs"/"marketing_agency_audiences"; out.mkdir(parents=True,exist_ok=True)
path=out/f"audiencias_agencia_ejecutivo_{datetime.now():%Y%m%d_%H%M%S}.xlsx"

def sheet(writer,name,df,title):
    df.to_excel(writer,sheet_name=name,startrow=3,index=False)
    wb=writer.book; ws=writer.sheets[name]; ws.hide_gridlines(2); ws.freeze_panes(4,0)
    fmt=wb.add_format({"bold":True,"font_size":16,"font_color":"#FFFFFF","bg_color":"#172554"})
    ws.merge_range(0,0,0,max(5,len(df.columns)-1),title,fmt); ws.set_row(0,26)
    ws.autofilter(3,0,3+len(df),max(0,len(df.columns)-1))
    pct=wb.add_format({"num_format":"0.0%"})
    for i,c in enumerate(df.columns):
        fmt=pct if any(t in str(c).lower() for t in ["tasa","participacion","aptitud","contactabilidad","ci95"]) else None
        ws.set_column(i,i,min(32,max(12,len(str(c))+2)),fmt)

with pd.ExcelWriter(path,engine="xlsxwriter",datetime_format="yyyy-mm-dd") as w:
    wb=w.book; ws=wb.add_worksheet("00_Resumen"); w.sheets["00_Resumen"]=ws; ws.hide_gridlines(2); ws.set_column("B:G",20)
    title=wb.add_format({"bold":True,"font_size":20,"font_color":"#FFFFFF","bg_color":"#172554"}); ws.merge_range("B2:G3","Audiencias Agencia Digital | Resumen",title)
    labels=[("B6","Personas",len(aud)),("D6","Tasa positiva",portfolio),("F6","HHI",hhi),("B9","Negativas RMKT",int(aud.negativa_sin_avance.sum())),("D9","Negativas activables",int(aud.negativa_activable.sum())),("F9","Aptitud externa",aud.apto_entrega_externa.mean())]
    k=wb.add_format({"bold":True,"font_size":15,"bg_color":"#EFF6FF","border":1}); l=wb.add_format({"font_color":"#475569"})
    for cell,label,val in labels: ws.write(cell,val,k); ws.write(f"{cell[0]}{int(cell[1:])+1}",label,l)
    sheet(w,"01_Canales",ch.reset_index(),"Canales — escala, calidad e incertidumbre")
    sheet(w,"02_Proyecto_Canal",pc,"Proyecto × canal")
    sheet(w,"03_Cohortes",coh,"Cohortes")
    sheet(w,"04_Hipotesis",pair,"Hipótesis vs benchmark")
    sheet(w,"10_Base_Positiva",base_pos,"Base positiva activable")
    sheet(w,"11_Base_Negativa",base_neg,"Base negativa RMKT activable")
print(f"Excel generado: {path} | positivas={len(base_pos):,} | negativas={len(base_neg):,}")


## 6. Lectura final
- No recomendar presupuesto solo por tasa observada: faltan inversión/costos y causalidad.
- Priorizar la combinación de escala, tasa, incertidumbre, proyecto y cohorte.
- Usar grupo control para medir lift incremental.
- No entregar a terceros filas con `apto_entrega_externa = FALSE`.
